# 🎙️ Audio Emotion Classification System

## 📌 Project Overview
This project implements an **Audio Emotion Classification System** based on speech feature extraction and Machine Learning classifiers. The model extracts **Mel-Frequency Cepstral Coefficients (MFCCs)** and spectral energy features from audio samples, mapping them to 8 fundamental human emotions standard in the **RAVDASS** (Ryerson Audio-Visual Database of Emotional Speech and Song) dataset:
- 😃 **Happy**
- 😢 **Sad**
- 😠 **Angry**
- 😨 **Fearful**
- 🤢 **Disgust**
- 😲 **Surprised**
- 😌 **Calm**
- 😐 **Neutral**

## 🛠️ Machine Learning Pipeline
1. **Audio Feature Extraction**: Extract 31 acoustic features per frame (MFCC energies, zero-crossing rate, RMS energy, spectral statistics).
2. **Data Preprocessing & Scaling**: Feature standardization via `StandardScaler`.
3. **Model Selection & Tuning**: K-Nearest Neighbors (KNN) classifier with $k$-hyperparameter selection ($k=1 \dots 15$) and baseline comparison with Random Forest.
4. **Evaluation**: Accuracy metrics, confusion matrix visualization, and classification report.

In [1]:
# Step 1: Import Required Libraries and append local path
import os
import sys
import numpy as np
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Ensure project folder is in Python path
current_dir = os.path.dirname(os.path.abspath('__file__'))
audio_proj_dir = os.path.join(current_dir, 'Audio Classification')
if os.path.exists(audio_proj_dir) and audio_proj_dir not in sys.path:
    sys.path.insert(0, audio_proj_dir)
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Import helper functions
from audio_classification import extract_audio_features, generate_synthetic_dataset, EMOTION_NAMES

print('All libraries loaded successfully!')

## 📊 Step 2: Dataset Loading & Feature Extraction
We load audio samples and extract MFCC & acoustic features.

In [2]:
# Generate synthetic dataset for demonstration (or load real RAVDASS WAV files)
X, y = generate_synthetic_dataset(num_samples=400)
print(f'Extracted Feature Matrix Shape: {X.shape}')
print(f'Target Labels Count:          {len(y)}')
print(f'Supported Classes:            {EMOTION_NAMES}')

## 🔀 Step 3: Train / Test Split & Feature Scaling

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train_scaled.shape[0]}')
print(f'Testing samples:  {X_test_scaled.shape[0]}')

## 🤖 Step 4: Model Training & Hyperparameter Tuning (KNN)

In [4]:
# Find optimal K for KNN
k_values = list(range(1, 16))
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_scores.append(accuracy_score(y_train, knn.predict(X_train_scaled)))
    test_scores.append(accuracy_score(y_test, knn.predict(X_test_scaled)))

# Plot accuracy vs K
plt.figure(figsize=(8, 4))
plt.plot(k_values, train_scores, label='Train Accuracy', marker='o')
plt.plot(k_values, test_scores, label='Test Accuracy', marker='s')
plt.xlabel('Value of K (Neighbors)')
plt.ylabel('Accuracy')
plt.title('KNN Hyperparameter Selection (Optimal K)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 📈 Step 5: Model Evaluation & Metrics

In [5]:
best_knn = KNeighborsClassifier(n_neighbors=5)
best_knn.fit(X_train_scaled, y_train)
knn_preds = best_knn.predict(X_test_scaled)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_preds = rf.predict(X_test_scaled)

print('=== KNN Classification Report ===')
print(classification_report(y_test, knn_preds))

cm = confusion_matrix(y_test, knn_preds, labels=EMOTION_NAMES)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTION_NAMES, yticklabels=EMOTION_NAMES)
plt.title('KNN Audio Emotion Confusion Matrix')
plt.xlabel('Predicted Emotion')
plt.ylabel('Actual Emotion')
plt.show()

## 💡 Step 6: Audio Inference Example
Testing emotion prediction on a new audio feature vector.

In [6]:
test_idx = 0
sample_feature = X_test_scaled[test_idx].reshape(1, -1)
expected_label = y_test[test_idx]
predicted_label = best_knn.predict(sample_feature)[0]
confidence = np.max(best_knn.predict_proba(sample_feature)[0]) * 100

print(f'Expected Emotion:  {expected_label}')
print(f'Predicted Emotion: {predicted_label} ({confidence:.1f}% confidence)')